# PRUDENCIA – Fine-Tuning de JuriBERT

Notebook pédagogique montrant un pipeline complet de Fine-Tuning avec **JuriBERT** pour une tâche de classification de textes juridiques.

## 1. Import des bibliothèques

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


## 2. Configuration

In [ ]:
MODEL_NAME = "dascim/juribert-base"
DATASET_PATH = Path("../datasets/deep_learning/prudencia_annotated_cases.csv")

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

MAX_LENGTH = 512
RANDOM_STATE = 42
TEST_SIZE = 0.20
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-5

OUTPUT_DIR = Path("outputs/juribert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Chargement du dataset

In [ ]:
df = pd.read_csv(DATASET_PATH, sep=None, engine="python")
display(df.head())
print(df.shape)


## 4. Nettoyage

In [ ]:
df = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
df = df[(df[TEXT_COLUMN]!="") & (df[LABEL_COLUMN]!="")]
display(df[LABEL_COLUMN].value_counts())


## 5. Encodage des classes

In [ ]:
encoder = LabelEncoder()
df["labels"] = encoder.fit_transform(df[LABEL_COLUMN])

id2label = {i:l for i,l in enumerate(encoder.classes_)}
label2id = {v:k for k,v in id2label.items()}

display(df.head())


## 6. Train / Validation

In [ ]:
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["labels"],
)

train_ds = Dataset.from_pandas(train_df[[TEXT_COLUMN,"labels"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[[TEXT_COLUMN,"labels"]], preserve_index=False)


## 7. Chargement de JuriBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)


## 8. Tokenisation

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

remove_cols=[c for c in train_ds.column_names if c not in ["input_ids","attention_mask","token_type_ids","labels"]]
train_ds=train_ds.remove_columns(remove_cols)
valid_ds=valid_ds.remove_columns(remove_cols)


## 9. Paramètres d'entraînement

In [ ]:
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR/"checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)


## 10. Fonction des métriques

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


## 11. Création du Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


## 12. Entraînement

In [ ]:
trainer.train()

## 13. Évaluation

In [ ]:
metrics = trainer.evaluate()
print(json.dumps(metrics, indent=4))


## 14. Sauvegarde

In [ ]:
trainer.save_model(OUTPUT_DIR/"final_model")
tokenizer.save_pretrained(OUTPUT_DIR/"final_model")


## 15. Exemple d'inférence

In [ ]:
texte = "Le système utilise des données biométriques pour identifier automatiquement des personnes."

inputs = tokenizer(
    texte,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

with torch.no_grad():
    outputs = model(**inputs)

probas = torch.softmax(outputs.logits, dim=-1)[0]
prediction = int(torch.argmax(probas))

print("Classe :", id2label[prediction])

for i,p in enumerate(probas):
    print(f"{id2label[i]} : {float(p):.4f}")


## Conclusion

Ce notebook présente un pipeline complet de Fine-Tuning avec JuriBERT :
1. chargement du dataset ;
2. préparation des données ;
3. tokenisation ;
4. entraînement ;
5. évaluation ;
6. sauvegarde ;
7. inférence.